# Notebook 52: The Master OED Destination Matrix

In this notebook, we enrich the Master Territorial Curriculum Catalog with the **Origin-Education-Destination (OED)** outcome metrics:
- Regional Income
- Track-level Gender Gaps (Mathematically derived from `hf_students_upper_sec_stat_2024_25.parquet`)
- Structural Risk proxies for Precarity/Shadow Economy


In [1]:
import pandas as pd
import numpy as np
import json
from pathlib import Path

ROOT = Path('c:/Users/Dell/Documents/Antigravity/Italienation').resolve()
PROC = ROOT / 'local_data/processed'

# 1. Load the Base Curriculum CSV
base_df = pd.read_csv(PROC / 'master_territorial_curriculum.csv')

# 2. Load Demographics to Calculate Actual Gender Gap per Track/SubTrack
students = pd.read_parquet(ROOT / 'local_data/UNICA/hf_students_upper_sec_stat_2024_25.parquet')
students['TIPOPERCORSO'] = students['TIPOPERCORSO'].fillna('Sconosciuto').str.title()
students['INDIRIZZO'] = students['INDIRIZZO'].fillna('Sconosciuto').str.title()

students['ALUNNIMASCHI'] = pd.to_numeric(students['ALUNNIMASCHI'], errors='coerce').fillna(0)
students['ALUNNIFEMMINE'] = pd.to_numeric(students['ALUNNIFEMMINE'], errors='coerce').fillna(0)

track_gender = students.groupby(['TIPOPERCORSO', 'INDIRIZZO'])[['ALUNNIMASCHI', 'ALUNNIFEMMINE']].sum().reset_index()
track_gender['Total'] = track_gender['ALUNNIMASCHI'] + track_gender['ALUNNIFEMMINE']
track_gender['Pct_Male'] = (track_gender['ALUNNIMASCHI'] / track_gender['Total'] * 100).round(1)
track_gender['Pct_Female'] = (track_gender['ALUNNIFEMMINE'] / track_gender['Total'] * 100).round(1)

# 3. Load Income Data
income = pd.read_csv(PROC / 'istat_household_income_by_region.csv')
if 'Region' in income.columns:
    income['REGIONE'] = income['Region'].str.title()
    base_df = pd.merge(base_df, income[['REGIONE', 'Median_Household_Income_EUR']], on='REGIONE', how='left')

# Group by to build hierarchical JSON
grouped = base_df.groupby(['REGIONE', 'PROVINCIA', 'DESCRIZIONECOMUNE', 'TIPOPERCORSO', 'INDIRIZZO'])['DISCIPLINA'].apply(list).reset_index()
if 'Median_Household_Income_EUR' in base_df.columns:
    grouped = pd.merge(grouped, base_df[['REGIONE', 'PROVINCIA', 'DESCRIZIONECOMUNE', 'TIPOPERCORSO', 'INDIRIZZO', 'Median_Household_Income_EUR']].drop_duplicates(), on=['REGIONE', 'PROVINCIA', 'DESCRIZIONECOMUNE', 'TIPOPERCORSO', 'INDIRIZZO'], how='left')
else:
    grouped['Median_Household_Income_EUR'] = np.nan

# Merge Gender Data
grouped = pd.merge(grouped, track_gender[['TIPOPERCORSO', 'INDIRIZZO', 'Pct_Male', 'Pct_Female']], on=['TIPOPERCORSO', 'INDIRIZZO'], how='left')

def get_structural_risk(tipo, ind):
    tipo = str(tipo).lower()
    if 'liceo' in tipo:
        return {"Academic_Failure_Risk": "Low", "Precarity_Risk": "Low (Univ Bound)"}
    elif 'tecnico' in tipo:
        return {"Academic_Failure_Risk": "Medium", "Precarity_Risk": "Medium"}
    else:
        return {"Academic_Failure_Risk": "High", "Precarity_Risk": "High (Black Labour Vulnerable)"}

catalog = {}

for row in grouped.itertuples(index=False):
    reg = str(row.REGIONE).title()
    prov = str(row.PROVINCIA).title()
    com = str(row.DESCRIZIONECOMUNE).title()
    tipo = str(row.TIPOPERCORSO)
    ind = str(row.INDIRIZZO)
    mat = sorted(list(set(row.DISCIPLINA)))
    income_val = row.Median_Household_Income_EUR
    pct_m = row.Pct_Male
    pct_f = row.Pct_Female
    
    if reg not in catalog:
        catalog[reg] = {
            "SocioEconomic_Context": {
                "Median_Income_EUR": income_val if pd.notna(income_val) else "N/A"
            },
            "Provinces": {}
        }
    
    if prov not in catalog[reg]["Provinces"]:
        catalog[reg]["Provinces"][prov] = {"Municipalities": {}}
        
    if com not in catalog[reg]["Provinces"][prov]["Municipalities"]:
        catalog[reg]["Provinces"][prov]["Municipalities"][com] = {"Tracks": {}}
        
    if tipo not in catalog[reg]["Provinces"][prov]["Municipalities"][com]["Tracks"]:
        catalog[reg]["Provinces"][prov]["Municipalities"][com]["Tracks"][tipo] = {}
        
    if ind not in catalog[reg]["Provinces"][prov]["Municipalities"][com]["Tracks"][tipo]:
        outcomes = get_structural_risk(tipo, ind)
        outcomes['Gender_Ratio_Pct_Male'] = pct_m if pd.notna(pct_m) else "N/A"
        outcomes['Gender_Ratio_Pct_Female'] = pct_f if pd.notna(pct_f) else "N/A"
        
        catalog[reg]["Provinces"][prov]["Municipalities"][com]["Tracks"][tipo][ind] = {
            "Subjects": mat,
            "Track_Outcomes": outcomes
        }

# Export to JSON
json_path = PROC / 'master_oed_destination_matrix.json'
with open(json_path, 'w', encoding='utf-8') as f:
    json.dump(catalog, f, ensure_ascii=False, indent=2)

print(f"Master OED JSON exported to {json_path}")


Master OED JSON exported to C:\Users\Dell\Documents\Antigravity\Italienation\local_data\processed\master_oed_destination_matrix.json
